# 🚗 VN-Traffic-Density — Stage 2: Finetune YOLOv8s trên BDD100K

**Pipeline:** COCO Pretrained → **BDD100K Finetune** → VN Data Finetune

**Mục tiêu giai đoạn này:**
- Thu hẹp domain gap từ *ảnh tổng quát* (COCO) sang *cảnh giao thông thực tế*
- Freeze 10 lớp backbone, chỉ train detection head
- 4 classes: `car (0)`, `truck (1)`, `bus (2)`, `motor (3)`

---
**Cấu trúc thư mục input (Google Drive):**
```
MyDrive/
└── bdd100k/
    ├── images/
    │   ├── train/   ← ảnh .jpg
    │   └── val/
    └── labels/
        ├── train/   ← .txt YOLO format, classes 0-3
        └── val/
```

## 0. Kiểm tra GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 1. Cài đặt thư viện

In [ ]:
!pip install ultralytics==8.3.* --quiet
!pip install supervision --quiet

import ultralytics
ultralytics.checks()

## 2. Mount Google Drive & khai báo đường dẫn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

# ─── CẤU HÌNH ĐƯỜNG DẪN ───────────────────────────────────────────────────────
DRIVE_ROOT   = Path('/content/drive/MyDrive')
BDD_ROOT     = DRIVE_ROOT / 'bdd100k'          # thư mục gốc dataset
PROJECT_DIR  = DRIVE_ROOT / 'vn_traffic_yolo'  # nơi lưu kết quả train
# ──────────────────────────────────────────────────────────────────────────────

# Kiểm tra dataset tồn tại
for p in [
    BDD_ROOT / 'images' / 'train',
    BDD_ROOT / 'images' / 'val',
    BDD_ROOT / 'labels' / 'train',
    BDD_ROOT / 'labels' / 'val',
]:
    status = '✅' if p.exists() else '❌ MISSING'
    count  = len(list(p.glob('*'))) if p.exists() else 0
    print(f'{status}  {p}  ({count} files)')

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
print(f'\n📁 Output dir: {PROJECT_DIR}')

## 3. Phân tích nhanh dataset

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

CLASSES = ['car', 'truck', 'bus', 'motor']
COLORS  = ['#4C9BE8', '#E87B4C', '#4CE87B', '#E84C9B']

def count_labels(label_dir: Path):
    counter = Counter()
    n_files = 0
    for txt in label_dir.glob('*.txt'):
        n_files += 1
        for line in txt.read_text().strip().splitlines():
            if line.strip():
                cls = int(line.split()[0])
                counter[cls] += 1
    return counter, n_files

print('── Đang đếm nhãn, vui lòng chờ... ──')
train_cnt, n_train = count_labels(BDD_ROOT / 'labels' / 'train')
val_cnt,   n_val   = count_labels(BDD_ROOT / 'labels' / 'val')

print(f'\nTập TRAIN: {n_train:,} ảnh')
for i, cls in enumerate(CLASSES):
    print(f'  {cls:>6}: {train_cnt[i]:>8,} instances')

print(f'\nTập VAL  : {n_val:,} ảnh')
for i, cls in enumerate(CLASSES):
    print(f'  {cls:>6}: {val_cnt[i]:>8,} instances')

# Biểu đồ phân phối
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (cnt, title) in zip(axes, [(train_cnt, 'Train'), (val_cnt, 'Val')]):
    vals = [cnt[i] for i in range(4)]
    bars = ax.bar(CLASSES, vals, color=COLORS, edgecolor='white', linewidth=0.8)
    ax.set_title(f'Class Distribution — {title}', fontsize=13, fontweight='bold')
    ax.set_ylabel('Số instances')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(vals)*0.01,
                f'{v:,}', ha='center', va='bottom', fontsize=9)
    ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(str(PROJECT_DIR / 'class_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Biểu đồ đã lưu vào project dir')

## 4. Tạo file cấu hình dataset (data.yaml)

In [ ]:
import yaml

data_yaml_path = PROJECT_DIR / 'bdd100k_traffic.yaml'

data_config = {
    'path'  : str(BDD_ROOT),
    'train' : 'images/train',
    'val'   : 'images/val',
    'nc'    : 4,
    'names' : {0: 'car', 1: 'truck', 2: 'bus', 3: 'motor'},

    # Metadata
    '_comment': 'Stage 2 — BDD100K finetune for VN-Traffic-Density project'
}

with open(data_yaml_path, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False, allow_unicode=True)

print('✅ data.yaml đã tạo:')
print(data_yaml_path.read_text())

## 5. Cấu hình Hyperparameter — Stage 2

In [ ]:
# ─── HYPERPARAMETERS — STAGE 2 (BDD100K FINETUNE) ────────────────────────────
# Theo thiết kế pipeline:
#   - Freeze 10 lớp backbone → chỉ train detection head & neck
#   - LR thấp (0.005) để không phá vỡ backbone đã học từ COCO
#   - 30 epochs là đủ vì chỉ adapt sang domain giao thông

STAGE2_CONFIG = dict(
    # ── Model & Data ──
    model       = 'yolov8s.pt',      # COCO pretrained, tự download
    data        = str(data_yaml_path),
    project     = str(PROJECT_DIR),
    name        = 'stage2_bdd100k',
    exist_ok    = True,

    # ── Training schedule ──
    epochs      = 30,
    patience    = 10,                 # early stopping
    batch       = 16,                 # điều chỉnh nếu OOM: 8 hoặc -1 (auto)
    imgsz       = 640,

    # ── Optimizer ──
    optimizer   = 'AdamW',
    lr0         = 0.005,              # LR thấp hơn scratch
    lrf         = 0.01,               # final LR = lr0 × lrf
    momentum    = 0.937,
    weight_decay= 0.0005,
    warmup_epochs = 3.0,

    # ── Freeze backbone ──
    freeze      = 10,                 # freeze 10 lớp đầu (backbone)

    # ── Augmentation — vừa phải cho stage 2 ──
    mosaic      = 1.0,
    mixup       = 0.0,                # tắt mixup ở stage 2
    hsv_h       = 0.015,
    hsv_s       = 0.7,
    hsv_v       = 0.3,
    degrees     = 0.0,
    translate   = 0.1,
    scale       = 0.5,
    fliplr      = 0.5,
    flipud      = 0.0,

    # ── Misc ──
    device      = 0,                  # GPU 0
    workers     = 4,
    cache       = False,              # True nếu RAM đủ (>16GB)
    save        = True,
    save_period = 5,                  # lưu checkpoint mỗi 5 epoch
    plots       = True,
    verbose     = True,
)

print('⚙️  Stage 2 Config:')
for k, v in STAGE2_CONFIG.items():
    print(f'   {k:<18} = {v}')

## 6. Kiểm tra 1 batch trước khi train

In [ ]:
import random
import cv2
from IPython.display import display
from PIL import Image
import matplotlib.patches as patches

def visualize_sample(img_dir: Path, lbl_dir: Path, n: int = 4):
    """Hiển thị n ảnh ngẫu nhiên với bounding boxes."""
    imgs = list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png'))
    if not imgs:
        print('Không tìm thấy ảnh trong thư mục'); return

    samples = random.sample(imgs, min(n, len(imgs)))
    fig, axes = plt.subplots(1, len(samples), figsize=(16, 4))
    if len(samples) == 1: axes = [axes]

    for ax, img_path in zip(axes, samples):
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        H, W = img.shape[:2]
        ax.imshow(img)

        lbl_path = lbl_dir / (img_path.stem + '.txt')
        if lbl_path.exists():
            for line in lbl_path.read_text().strip().splitlines():
                parts = line.strip().split()
                if len(parts) < 5: continue
                cls, cx, cy, bw, bh = int(parts[0]), *map(float, parts[1:5])
                x1 = (cx - bw/2) * W
                y1 = (cy - bh/2) * H
                rect = patches.Rectangle(
                    (x1, y1), bw*W, bh*H,
                    linewidth=2, edgecolor=COLORS[cls], facecolor='none'
                )
                ax.add_patch(rect)
                ax.text(x1, y1-2, CLASSES[cls], color=COLORS[cls],
                        fontsize=7, fontweight='bold')

        ax.set_title(img_path.name[:20], fontsize=8)
        ax.axis('off')

    plt.suptitle('Sample Train Images với Annotations', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

visualize_sample(
    BDD_ROOT / 'images' / 'train',
    BDD_ROOT / 'labels' / 'train',
    n=4
)

## 7. 🚀 BẮT ĐẦU TRAIN — Stage 2

In [ ]:
from ultralytics import YOLO
import time

print('=' * 60)
print('  STAGE 2: FINETUNE YOLOv8s trên BDD100K')
print('  COCO pretrained → freeze backbone 10 lớp')
print('=' * 60)

# Load model COCO pretrained
model = YOLO(STAGE2_CONFIG['model'])

print(f'\n✅ Loaded: {STAGE2_CONFIG["model"]}')
print(f'   Backbone layers: {sum(1 for _ in model.model.model[:10])} lớp sẽ bị freeze')

t0 = time.time()

results = model.train(**STAGE2_CONFIG)

elapsed = time.time() - t0
print(f'\n⏱️  Tổng thời gian train: {elapsed/3600:.2f} giờ ({elapsed/60:.1f} phút)')

## 8. Đánh giá kết quả trên tập Val

In [ ]:
# Đường dẫn best weights
best_weights = PROJECT_DIR / 'stage2_bdd100k' / 'weights' / 'best.pt'
last_weights = PROJECT_DIR / 'stage2_bdd100k' / 'weights' / 'last.pt'

print(f'Best weights: {best_weights}')
print(f'Exists      : {best_weights.exists()}')

# Load best model và validate
model_best = YOLO(str(best_weights))

val_results = model_best.val(
    data     = str(data_yaml_path),
    imgsz    = 640,
    batch    = 16,
    device   = 0,
    split    = 'val',
    save_json= False,
    plots    = True,
    verbose  = True,
)

print('\n' + '='*50)
print('  KẾT QUẢ ĐÁNH GIÁ STAGE 2')
print('='*50)
print(f'  mAP@0.5       : {val_results.box.map50:.4f}  (mục tiêu: ≥ 0.80 ở stage này)')
print(f'  mAP@0.5:0.95  : {val_results.box.map:.4f}')
print(f'  Precision     : {val_results.box.mp:.4f}')
print(f'  Recall        : {val_results.box.mr:.4f}')
print()

# Per-class breakdown
print('  Per-class mAP@0.5:')
for i, (cls_name, ap) in enumerate(zip(CLASSES, val_results.box.ap50)):
    bar = '█' * int(ap * 30)
    print(f'    {cls_name:>6}  {bar:<30}  {ap:.4f}')

## 9. Vẽ training curves

In [ ]:
import pandas as pd

results_csv = PROJECT_DIR / 'stage2_bdd100k' / 'results.csv'

if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle('Stage 2 — Training Curves (BDD100K)', fontsize=14, fontweight='bold')

    METRICS = [
        ('train/box_loss',   'Train Box Loss',   '#E84C4C'),
        ('train/cls_loss',   'Train Cls Loss',   '#E8944C'),
        ('train/dfl_loss',   'Train DFL Loss',   '#E8D44C'),
        ('metrics/mAP50(B)', 'mAP@0.5',          '#4CE87B'),
        ('metrics/precision(B)', 'Precision',    '#4C9BE8'),
        ('metrics/recall(B)',    'Recall',        '#9B4CE8'),
    ]

    for ax, (col, title, color) in zip(axes.flatten(), METRICS):
        if col in df.columns:
            ax.plot(df['epoch'], df[col], color=color, linewidth=2)
            ax.set_title(title, fontsize=11)
            ax.set_xlabel('Epoch')
            ax.grid(True, alpha=0.3)
            ax.spines[['top','right']].set_visible(False)
            # Annotate best value
            best_val = df[col].max() if 'loss' not in col else df[col].min()
            best_ep  = df.loc[df[col] == best_val, 'epoch'].values[0]
            ax.axvline(best_ep, color=color, linestyle='--', alpha=0.5)
            ax.set_title(f'{title}\nbest={best_val:.4f} @ ep{best_ep:.0f}', fontsize=10)
        else:
            ax.text(0.5, 0.5, f'{col}\nnot found', ha='center', va='center', transform=ax.transAxes)

    plt.tight_layout()
    save_path = PROJECT_DIR / 'training_curves_stage2.png'
    plt.savefig(str(save_path), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✅ Training curves lưu tại: {save_path}')
else:
    print('⚠️  results.csv chưa tồn tại — hãy chạy cell train trước')

## 10. Inference thử trên ảnh Val — kiểm tra visual

In [ ]:
import random

val_imgs = list((BDD_ROOT / 'images' / 'val').glob('*.jpg'))
test_batch = random.sample(val_imgs, min(6, len(val_imgs)))

preds = model_best.predict(
    source    = test_batch,
    imgsz     = 640,
    conf      = 0.35,
    iou       = 0.45,
    device    = 0,
    verbose   = False,
)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Stage 2 — Inference Preview (Val Set, conf=0.35)', fontsize=13, fontweight='bold')

for ax, r in zip(axes.flatten(), preds):
    img = r.plot(line_width=2, font_size=10)  # BGR numpy
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    cls_counts = Counter(int(c) for c in r.boxes.cls.cpu()) if r.boxes else Counter()
    summary = '  '.join(f'{CLASSES[k]}:{v}' for k, v in sorted(cls_counts.items()))
    ax.set_title(summary if summary else 'no detection', fontsize=9)
    ax.axis('off')

plt.tight_layout()
save_path = PROJECT_DIR / 'inference_preview_stage2.png'
plt.savefig(str(save_path), dpi=120, bbox_inches='tight')
plt.show()
print(f'✅ Preview lưu tại: {save_path}')

## 11. Tóm tắt & chuẩn bị sang Stage 3

In [ ]:
import shutil, json

# Lưu summary JSON để Stage 3 đọc lại
summary = {
    'stage'      : 2,
    'base_model' : 'yolov8s.pt (COCO)',
    'dataset'    : 'BDD100K — 4 traffic classes',
    'epochs_trained': int(df['epoch'].max()) if results_csv.exists() else STAGE2_CONFIG['epochs'],
    'best_weights': str(best_weights),
    'metrics': {
        'mAP50'     : round(float(val_results.box.map50), 4),
        'mAP50_95'  : round(float(val_results.box.map),   4),
        'precision' : round(float(val_results.box.mp),    4),
        'recall'    : round(float(val_results.box.mr),    4),
    },
    'next_stage' : {
        'description' : 'Finetune trên dữ liệu tự thu thập tại Đà Nẵng',
        'freeze'      : 0,
        'lr0'         : 0.001,
        'epochs'      : 100,
        'patience'    : 20,
    }
}

summary_path = PROJECT_DIR / 'stage2_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print('╔══════════════════════════════════════════════════╗')
print('║           STAGE 2 — HOÀN THÀNH ✅               ║')
print('╠══════════════════════════════════════════════════╣')
print(f'║  mAP@0.5      : {val_results.box.map50:.4f}                        ║')
print(f'║  mAP@0.5:0.95 : {val_results.box.map:.4f}                        ║')
print(f'║  Precision    : {val_results.box.mp:.4f}                        ║')
print(f'║  Recall       : {val_results.box.mr:.4f}                        ║')
print('╠══════════════════════════════════════════════════╣')
print(f'║  Best weights : .../weights/best.pt              ║')
print(f'║  Summary JSON : stage2_summary.json              ║')
print('╠══════════════════════════════════════════════════╣')
print('║  BƯỚC TIẾP THEO — Stage 3:                       ║')
print('║  Dùng best.pt → finetune trên data Đà Nẵng       ║')
print('║  freeze=0, lr0=0.001, epochs=100                 ║')
print('╚══════════════════════════════════════════════════╝')